In [0]:
#Qn.Write a pyspark code to create data validation between source and target table

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("DataValidation") \
    .getOrCreate()

# Sample source data (for demonstration)
source_data = [
    (1, "Jeff", "Doe", 5000),
    (2, "Rose", "Smith", 6000),
    (3, "Phil", "Johnson", 7000),
]

# Sample target data (for demonstration)
target_data = [
    (1, "Jeff", "Doe", 5000),
    (2, "Rose", "Smith", 6000),
    (3, "Phil", "Johnson", 7000),
]

# Define schema for source and target tables
columns = ["id", "first_name", "last_name", "salary"]

# Create DataFrames for source and target tables
source_df = spark.createDataFrame(source_data, columns)
target_df = spark.createDataFrame(target_data, columns)

# Data Validation Step 1: Check if record counts match
source_count = source_df.count()
target_count = target_df.count()

print(f"Source Record Count: {source_count}")
print(f"Target Record Count: {target_count}")

if source_count != target_count:
    print("Validation Failed: Record counts do not match.")
else:
    print("Record counts match.")

# Data Validation Step 2: Check for missing records in the target
missing_in_target = source_df.subtract(target_df)
print("Missing records in target:")
missing_in_target.show()

# Data Validation Step 3: Check for extra records in the target (present in target but not in source)
extra_in_target = target_df.subtract(source_df)
print("Extra records in target:")
extra_in_target.show()

# Data Validation Step 4: Check if all fields match (row-by-row comparison)
# We can do a full comparison for all columns in both source and target
mismatched_data = source_df.join(target_df, on=["id"], how="left_anti")  # Find rows in source not in target
print("Mismatched records between source and target:")
mismatched_data.show()

# Data Validation Step 5: Additional field-wise comparison (optional)
# This will compare specific columns to check if they match across source and target
field_comparison = source_df.join(target_df, on=["id"]) \
    .filter(
        (source_df.first_name != target_df.first_name) |
        (source_df.last_name != target_df.last_name) |
        (source_df.salary != target_df.salary)
    )
print("Field mismatches between source and target:")
field_comparison.show()


Source Record Count: 3
Target Record Count: 3
Record counts match.
Missing records in target:
+---+----------+---------+------+
| id|first_name|last_name|salary|
+---+----------+---------+------+
+---+----------+---------+------+

Extra records in target:
+---+----------+---------+------+
| id|first_name|last_name|salary|
+---+----------+---------+------+
+---+----------+---------+------+

Mismatched records between source and target:
+---+----------+---------+------+
| id|first_name|last_name|salary|
+---+----------+---------+------+
+---+----------+---------+------+

Field mismatches between source and target:
+---+----------+---------+------+----------+---------+------+
| id|first_name|last_name|salary|first_name|last_name|salary|
+---+----------+---------+------+----------+---------+------+
+---+----------+---------+------+----------+---------+------+

